In [ ]:
# EDA notebook cell
import os, glob, math
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
DATA = Path("./hb_yolo_singlecls")
splits = ["train","val"]

rows = []
for split in splits:
    img_dir, lbl_dir = DATA/split/"images", DATA/split/"labels"
    for txt in lbl_dir.glob("*.txt"):
        stem = txt.stem
        # find image
        imgp = None
        for ext in (".jpg",".jpeg",".png",".bmp",".tif",".tiff"):
            p = img_dir/f"{stem}{ext}"
            if p.exists(): imgp=p; break
        if imgp is None: continue
        with Image.open(imgp) as im:
            w,h = im.size
        with open(txt) as f:
            for ln in f:
                ps = ln.strip().split()
                if len(ps)!=5: continue
                c, xc, yc, bw, bh = int(ps[0]), *map(float, ps[1:])
                rows.append({
                    "split": split, "file": stem, "img_w": w, "img_h": h,
                    "class": c, "xc": xc, "yc": yc, "bw": bw, "bh": bh,
                    "box_area_norm": bw*bh, "aspect": (bw/(bh+1e-9))
                })

In [ ]:
df = pd.DataFrame(rows)
print(df.head())
print("\nCounts per split:", df.groupby("split").size().to_dict())
print("Boxes per image (train):")
print(df[df.split=="train"].groupby("file").size().describe())
print("\nClass counts:", df["class"].value_counts())

In [ ]:
# Box area histogram
plt.figure()
df["box_area_norm"].hist(bins=50)
plt.title("Normalized box area distribution")
plt.xlabel("bw*bh (normalized)"); plt.ylabel("count")
plt.show()

In [ ]:
# Aspect ratio histogram
plt.figure()
np.clip(df["aspect"].values, 0, 10)
df["aspect"].clip(0,10).hist(bins=50)
plt.title("BBox aspect ratio (w/h)"); plt.xlabel("ratio (clipped)"); plt.ylabel("count")
plt.show()

In [ ]:
# Image dimension distribution
plt.figure()
(df["img_w"]/df["img_h"]).clip(0,4).hist(bins=50)
plt.title("Image aspect ratio distribution")
plt.xlabel("w/h"); plt.ylabel("count"); plt.show()

In [ ]:
# Heatmap of box centers
heat = np.zeros((50,50), dtype=np.int32)
xs = (df["xc"]*49).astype(int).values
ys = (df["yc"]*49).astype(int).values
for x,y in zip(xs,ys):
    if 0<=x<50 and 0<=y<50:
        heat[y,x]+=1
plt.figure()
plt.imshow(heat, origin="lower")
plt.title("BBox center heatmap (50x50)"); plt.colorbar(); plt.show()


In [ ]:
# visualize a few images + labels
import random
from ultralytics.utils.plotting import Annotator, colors

In [ ]:


def show_samples(split="train", n=6):
    img_dir = DATA/split/"images"
    lbl_dir = DATA/split/"labels"
    candidates = list(lbl_dir.glob("*.txt"))
    random.shuffle(candidates)
    for txt in candidates[:n]:
        stem = txt.stem
        for ext in (".jpg",".jpeg",".png",".bmp",".tif",".tiff"):
            imgp = img_dir/f"{stem}{ext}"
            if imgp.exists(): break
        im = np.array(Image.open(imgp).convert("RGB"))
        h,w = im.shape[:2]
        ann = Annotator(im.copy())
        with open(txt) as f:
            for ln in f:
                c, xc, yc, bw, bh = ln.strip().split()
                c = int(c); xc=float(xc); yc=float(yc); bw=float(bw); bh=float(bh)
                x1 = (xc - bw/2)*w; y1 = (yc - bh/2)*h
                x2 = (xc + bw/2)*w; y2 = (yc + bh/2)*h
                ann.box_label([x1,y1,x2,y2], f"{c}", color=colors(c,True))
        plt.figure(figsize=(6,6)); plt.imshow(ann.result()); plt.axis("off"); plt.show()



In [ ]:
show_samples("train", n=4)